In [ ]:
########## This code is used to identify river surface temperature profiles with major changes in downstream temperatures within CONUS 2013-2024 ##########

In [ ]:
# Import Libraries
import pandas as pd
import geopandas as gpd
import glob
import os
import numpy as np
import scipy.stats as stats
from scipy.stats import kruskal
from tqdm.notebook import tqdm

In [ ]:
#############################################
########### Pull in the initial datasets ############
############################################

In [ ]:
### Pull in the Temperature Data -- Up/Ds FIltering ###
# Set up the location of the data
FilePath = r"F:\Insert_File_Path_for_Folder_of_Snapped_Temperatures_for_Each_Dam" ## Temperatures for each dam # Update this file path

# Get the List of Monthly Data
CSVFiles = glob.glob(os.path.join(FilePath, "*Avg_Img_Temps.csv")) 

# Loop through the files for each dam and make one dataframe
All_Mo_Avg = pd.DataFrame()
for i in tqdm(range(len(CSVFiles))):
    try:
        x = pd.read_csv(CSVFiles[i], engine='python')
        All_Mo_Avg = pd.concat([All_Mo_Avg,x],axis=0)
    except pd.errors.EmptyDataError:
        print(CSVFiles[i], " is empty and has been skipped.") # In case some of the CSV files are empty
# Preview the data
All_Mo_Avg

In [ ]:
## Looking at just the wide points, and non dam points
All_Mo_Avg_Subset = All_Mo_Avg[(All_Mo_Avg['Avg_RWC_Wid'] >= 100) & (All_Mo_Avg['Up_Ds']!= 'Dam')]
All_Mo_Avg_Subset

In [ ]:
###  Pull in the Dam File ###
Dams = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile_with_Dam_Locations.shp") # Update this file path
## This Shapefile ^^ has all the dams used to pull temperatures in it with infromation from HILARRI matched to it (completed in ArcGIS) ##
Dams

In [ ]:
#### Clean up the Data ####
# Get Dam Info on the nodes/temperature Values  and clean up the dataset ##
HydroPower = pd.merge(All_Mo_Avg_Subset, Dams[['grod_id', 'dataset']], left_on = 'Assgn_dam', right_on = 'grod_id' , how='left').drop(columns = [ 'grod_id'])

# Remove Unwanted Columns
HydroDams_Clean = HydroPower.drop(columns=['Unnamed: 0'])

# Make an easier code for hydropower variables -- Updating the dictionary for simplicity
damtype_dict = {'Hydropower dam associated with power plant; no reservoir': 'HDNR', 'Hydropower dam associated with reservoir and power plant': 'HDR', 'Hydropower dam only; no reservoir or power plant': 'HDNR','Hydropower dam associated with reservoir; no power plant':'HDR', None: 'NoHydro'} # dictionary
HydroDams_Clean['HydroCode'] = HydroDams_Clean['dataset'].map(damtype_dict)  # apply dictionary

# Group datata into distance bins
bins = np.arange(-100, 100, 2).tolist() # create 2km bins
HydroDams_Clean['Bins'] = pd.cut(HydroDams_Clean['Dam_Dist_km'], bins) # apply bins

# Create a Code to easily group based on Up/Ds and Lake Flag
HydroDams_Clean['Up_Ds_Grp']  = np.nan
HydroDams_Clean['Up_Ds_Grp'] = np.where((HydroDams_Clean['Up_Ds'] == 'Downstream') & (HydroDams_Clean['lakeflag'] == 0) , 'Downstream River', HydroDams_Clean['Up_Ds_Grp'])
HydroDams_Clean['Up_Ds_Grp'] = np.where((HydroDams_Clean['Up_Ds'] == 'Downstream') & (HydroDams_Clean['lakeflag'] > 0) , 'Downstream Reservoir', HydroDams_Clean['Up_Ds_Grp'])
HydroDams_Clean['Up_Ds_Grp'] = np.where((HydroDams_Clean['Up_Ds'] == 'Upstream') & (HydroDams_Clean['lakeflag'] == 0) , 'Upstream River', HydroDams_Clean['Up_Ds_Grp'])
HydroDams_Clean['Up_Ds_Grp'] = np.where((HydroDams_Clean['Up_Ds'] == 'Upstream') & (HydroDams_Clean['lakeflag'] > 0) , 'Upstream Reservoir', HydroDams_Clean['Up_Ds_Grp'])

# Preview Data
HydroDams_Clean

In [ ]:
###########################################################
############## FILTER TO UNIQUE, USABLE PROFILES ##############
###########################################################

In [ ]:
## Get Profiles -- Where there are river nodes upstream ##
UpRiver= HydroDams_Clean[(HydroDams_Clean["Up_Ds_Grp"] == "Upstream River")]
UpRiver_grp = UpRiver.groupby(['Month','Day','Year','Assgn_dam']).agg({'Join_Node': ['count']})
UpRiver_grp.columns = ["NodeCount"]
UpRiver_grp = UpRiver_grp.reset_index()

## Identify Profiles with 5 Upstream River Nodes
DamTemps_RivUp = UpRiver_grp[UpRiver_grp["NodeCount"]>= 5]

## Get Profiles -- Where there are river nodes directly downstream from the dam ##
DsRiver= HydroDams_Clean[(HydroDams_Clean["Up_Ds_Grp"] == "Downstream River") & (HydroDams_Clean["Dam_Dist_km"] <= 20)]
DsRiver_grp = DsRiver.groupby(['Month','Day','Year','Assgn_dam']).agg({'Join_Node': ['count']})
DsRiver_grp.columns = ["NodeCount"]
DsRiver_grp = DsRiver_grp.reset_index()

## Identify Profiles with 5 Downstream River Nodes
DamTemps_RivDs = DsRiver_grp[DsRiver_grp["NodeCount"]>= 5]

#### NUMBER OF PROFILES ####
UpAnDown = pd.merge(DamTemps_RivDs, DamTemps_RivUp, on = ['Assgn_dam', "Month", "Day", "Year"] , how='inner')
UpAnDown

In [ ]:
## Save the unique profiles with enough points to a CSV
UpAnDown.to_csv(r"F:\Insert_File_Output_Path\List_of_Profiles.csv") # Update this file path

In [ ]:
####################################################################
############ Identify Profiles with Significant Up/DS Differences ##############
####################################################################

In [ ]:
### Define a function to run the Kruskal Wallis tests iteratively ###

def kruskal_wallis_monte_carlo(df, col1_name, col2_name, num_simulations=1000):
    """
    A function to perform a Monte Carlo simulation for the Kruskal-Wallis test on two dataframe columns
    
    Args:
        df: The input dataframe
        col1_name (str): The name of the value column
        col2_name (str): The name of the group column
        num_simulations (int): The number of Monte Carlo iterations.

    Returns:
        float: The estimated p-value from the Monte Carlo simulation.
    """
    
    # Get the data for the two groups
    group1_data = df[df[col2_name] == df[col2_name].unique()[0]][col1_name].values
    group2_data = df[df[col2_name] == df[col2_name].unique()[1]][col1_name].values
    all_data = np.concatenate([group1_data, group2_data])
    n1 = len(group1_data)
    n2 = len(group2_data)

    # Calculate the observed H-statistic
    observed_h_statistic, _ = kruskal(group1_data, group2_data)
    
    # List to store H-statistics from simulations
    simulated_h_statistics = []

    # Run the Monte Carlo simulation
    for _ in range(num_simulations):
        # Permute the data under the null hypothesis (that samples are from the same distribution)
        permuted_data = np.random.permutation(all_data)
        permuted_group1 = permuted_data[:n1]
        permuted_group2 = permuted_data[n1:]
        
        # Calculate H-statistic for the permuted data
        h_statistic, _ = kruskal(permuted_group1, permuted_group2)
        simulated_h_statistics.append(h_statistic)

    # Calculate the p-value: the proportion of simulated H-statistics >= observed H-statistic
    p_value = np.sum(np.array(simulated_h_statistics) >= observed_h_statistic) / num_simulations
    
    return p_value, observed_h_statistic, simulated_h_statistics

In [ ]:
## Get Profile Points (filtering only 20km DS)##
DamTemps_Riv20 = HydroDams_Clean[(HydroDams_Clean["Dam_Dist_km"] <= 20)]

## Calculate Significance (Up River/20km)##
col_names =  ['Assgn_dam','Month','Day','Year', 'KW_Pval', 'MC_Pval']
KW_MC_Prof_Results_Riv20  = pd.DataFrame(columns = col_names)
Unique_Profiles_Riv20 =  UpAnDown[["Assgn_dam", "Month","Day", "Year"]]

for i in tqdm(range(len(Unique_Profiles_Riv20))):  # For selecting -- .iloc[i, 0] ## i for row , Dam Number (0), Month (1), Day (2), Year(3)
    ## Traditional KW Test ##
        # Set up the subsets for comparison -- All
    KruskalTest_1 = DamTemps_Riv20[(DamTemps_Riv20['Assgn_dam'] == Unique_Profiles_Riv20.iloc[i, 0]) & (DamTemps_Riv20['Month'] == Unique_Profiles_Riv20.iloc[i, 1]) & (DamTemps_Riv20['Day'] == Unique_Profiles_Riv20.iloc[i, 2]) & (DamTemps_Riv20['Year'] == Unique_Profiles_Riv20.iloc[i, 3]) & (DamTemps_Riv20['Up_Ds_Grp'] == "Downstream River")].Avg_Temp.tolist() # Downstream
    KruskalTest_2 = DamTemps_Riv20[(DamTemps_Riv20['Assgn_dam'] == Unique_Profiles_Riv20.iloc[i, 0]) & (DamTemps_Riv20['Month'] == Unique_Profiles_Riv20.iloc[i, 1]) & (DamTemps_Riv20['Day'] == Unique_Profiles_Riv20.iloc[i, 2]) & (DamTemps_Riv20['Year'] == Unique_Profiles_Riv20.iloc[i, 3]) & (DamTemps_Riv20['Up_Ds_Grp'] == "Upstream River")].Avg_Temp.tolist() # Upstream    
     
    # Run the test -- River vs DS
    h_statistic, p_value = stats.kruskal(KruskalTest_2,KruskalTest_1)

    ## Run the Monte Carlo Simulation
    KruskalTest_MC = DamTemps_Riv20[(DamTemps_Riv20['Assgn_dam'] == Unique_Profiles_Riv20.iloc[i, 0]) & (DamTemps_Riv20['Month'] == Unique_Profiles_Riv20.iloc[i, 1]) & (DamTemps_Riv20['Day'] == Unique_Profiles_Riv20.iloc[i, 2]) & (DamTemps_Riv20['Year'] == Unique_Profiles_Riv20.iloc[i, 3])]
    KruskalTest_MC = KruskalTest_MC[KruskalTest_MC['Up_Ds_Grp'].isin(['Upstream River', 'Downstream River'])]

    ## Run Monte Carlo 
    p_value_mc, observed_h_mc, h_stats_mc = kruskal_wallis_monte_carlo(KruskalTest_MC, col1_name='Avg_Temp', col2_name='Up_Ds_Grp', num_simulations=10000)

        # Get Dictionary
    dictionary = {'Assgn_dam': Unique_Profiles_Riv20.iloc[i, 0], 'Month': Unique_Profiles_Riv20.iloc[i, 1], 'Day': Unique_Profiles_Riv20.iloc[i, 2], 'Year': Unique_Profiles_Riv20.iloc[i, 3], 'KW_Pval': [p_value], 'MC_Pval': [p_value_mc]}
    df_dictionary = pd.DataFrame.from_dict(dictionary)
    
    # Add to DF
    output = pd.concat([KW_MC_Prof_Results_Riv20, df_dictionary], ignore_index=True)
    
    KW_MC_Prof_Results_Riv20 = output

In [ ]:
# Define Significant profiles (Riv Up/20km)
KW_MC_Prof_Results_Riv20['KW_Sig_05'] = np.where((KW_MC_Prof_Results_Riv20['KW_Pval'] <= 0.05), 'Significant', 'Not Significant')
KW_MC_Prof_Results_Riv20['MC_Sig_05'] = np.where((KW_MC_Prof_Results_Riv20['MC_Pval'] <= 0.05), 'Significant', 'Not Significant')
KW_MC_Prof_Results_Riv20

In [ ]:
## Save File ##
KW_MC_Prof_Results_Riv20.to_csv(r"F:\Insert_File_Output_for_Significance_Results.csv") # Update this file path

In [ ]:
print("Number of usable profiles: " + str(len(Unique_Profiles_Riv20)))

In [ ]:
MC_Signif_Profiles  = KW_MC_Prof_Results_Riv20[KW_MC_Prof_Results_Riv20['MC_Sig_05'] == 'Significant']
print("Number of 'significant profiles' from Monte Carlo simulation: " + str(len(MC_Signif_Profiles)))
print("Pecent Significant: " + str((len(MC_Signif_Profiles))/(len(KW_MC_Prof_Results_Riv20))* 100) + ' %')

In [ ]:
Differing_Signifs = KW_MC_Prof_Results_Riv20[((KW_MC_Prof_Results_Riv20['KW_Sig_05'] == 'Significant') & (KW_MC_Prof_Results_Riv20['MC_Sig_05'] == 'Not Significant')) | ((KW_MC_Prof_Results_Riv20['KW_Sig_05'] == 'Not Significant')& (KW_MC_Prof_Results_Riv20['MC_Sig_05'] == 'Significant'))]

print("Number of profiles with differing significances between the orginal KW test and Monte Carlo simulation: " + str(len(Differing_Signifs)))
print("Pecent Different: " + str((len(Differing_Signifs))/(len(KW_MC_Prof_Results_Riv20))* 100) + ' %')